# Descarga automatizada de datos del TFM (local, sin downscaling)

Este notebook está pensado para ejecutarse **en VS Code/Jupyter local**, no en Google Colab. Genera tres productos principales:

1. `dataset_landsat_sin_downscaling.csv`: Landsat 8/9 en la malla configurable (100 m por defecto), variables estáticas, meteorología ERA5-Land y morfología urbana.
2. `dataset_sentinel3_1km_sin_downscaling.csv`: serie diaria Sentinel-3 SLSTR L2 LST a 1 km, próxima a la hora Landsat, más auxiliares.
3. `dataset_pares_landsat_sentinel3.csv`: coincidencias del mismo día y con hasta dos horas de diferencia para entrenar downscaling.

No entrena modelos ni hace downscaling. Procesa el intervalo seleccionado y reutiliza caché para poder reanudar.

> Importante: el notebook anterior trataba la banda Sentinel-3 `F1` como LST. `F1` es una banda de incendios activos de nivel 1. Aquí se usa el producto L2 LST oficial y se filtran excepciones y nubes.

## 0. Preparación local

1. En VS Code, selecciona un kernel Python 3.10 o posterior.
2. Ejecuta la celda de instalación una sola vez y reinicia el kernel si VS Code lo solicita.
3. Edita únicamente las dos celdas siguientes: **CONFIGURACIÓN** y **CREDENCIALES**.
4. Ejecuta de arriba abajo.

Antes de usar ERA5-Land debes aceptar manualmente sus condiciones de uso en la página del dataset de CDS. Earth Engine abrirá el navegador la primera vez y guardará la autorización localmente.

**Propósito científico.** Este bloque instala las dependencias científicas y geoespaciales necesarias en el kernel local. Se ejecuta una sola vez y no altera las decisiones metodológicas del pipeline.


In [ ]:
# Ejecutar una sola vez en el kernel local. No usa comandos de Colab.
%pip install "earthengine-api>=1.5" "pandas>=2.1" "numpy>=1.26" "requests>=2.31" \
    "pyproj>=3.6" "shapely>=2.0" "geopandas>=0.14" "rasterio>=1.3" \
    "rasterstats>=0.19" "osmnx>=2.0" "xarray>=2024.1" "netcdf4>=1.6" \
    "h5netcdf>=1.3" "cdsapi>=0.7.7" "tqdm>=4.66" "folium>=0.16"

## 1. CONFIGURACIÓN — edita aquí el área, fechas y discretización

**Propósito científico.** Aquí se define el experimento: periodo, ROI, mallas, controles de calidad y salidas. Estos parámetros determinan el dominio espacial y temporal de todas las adquisiciones posteriores.


In [ ]:
from pathlib import Path

# -------------------- PERIODO --------------------
START_DATE = "2013-05-01"  # Landsat comienza aquí
END_DATE = "2025-10-31"    # fecha final inclusiva
ALLOWED_MONTHS = [5, 6, 7, 8, 9, 10]
DATE_WINDOWS = [(START_DATE, END_DATE)]  # El wrapper puede inyectar varias ventanas anuales exactas.

# -------------------- ÁREA DE ESTUDIO --------------------
CENTER_LAT = 37.4035
CENTER_LON = -5.9810
HALF_SIDE_M = 750             # ROI cuadrado: lado total = 2 * HALF_SIDE_M
ROI_ROTATION_DEG = 0.0        # giro antihorario del cuadrado y su malla
PROJECTED_CRS = "EPSG:25830" # ETRS89 / UTM 30N, adecuado para Sevilla
S3_REQUEST_CRS = "EPSG:32630" # UTM exigido por Sentinel-3 SLSTR en esta zona

# -------------------- MALLAS --------------------
NODE_SPACING_M = 100          # discretización modificable de los nodos
SENTINEL3_NATIVE_M = 1000     # tamaño exacto de los píxeles pedidos a Sentinel Hub
S3_NODE_AGG_RADIUS_M = 750    # nodos finos usados para resumir cada píxel S3

# Se incluye junto al notebook el CSV recuperado de la ejecución original (270 nodos).
# Cambia a None para generar una malla métrica nueva con NODE_SPACING_M.
NODES_CSV = Path.cwd() / "nodos_originales_100m.csv"

# -------------------- VARIABLES --------------------
# El núcleo mantiene exactamente las 23 columnas históricas. Añade aquí
# variables del catálogo para incorporarlas como columnas opcionales.
REQUESTED_VARIABLES = [
    "LST", "NDVI", "NDBI", "ALBEDO", "air_temperature",
    "relative_humidity", "solar_radiation", "wind_speed", "rain_3d",
    "elevation", "aspect_ratio", "buildings",
]

# -------------------- FUENTES Y CALIDAD --------------------
MAX_LANDSAT_CLOUD = 80
MAX_SENTINEL2_CLOUD = 80
S2_MAX_DAY_DIFFERENCE = 20
S3_MORNING_START_UTC = 7
S3_MORNING_END_UTC = 13
S3_TARGET_HOUR_UTC = 10
S3_REFERENCE_TOLERANCE_DAYS = 3
S3_PAIR_MAX_HOUR_DIFFERENCE = 2.0
S3_MAX_UNCERTAINTY_K = 5.0
INCLUDE_URBAN_MORPHOLOGY = True  # CNIG + OSM -> aspect_ratio, has_buildings
ERA5_SOURCE = "earth_engine"   # CDS se usa solo como fallback si se configura su token

# -------------------- SALIDAS Y REANUDACIÓN --------------------
OUTPUT_DIR = Path.cwd() / "data_downloads"
CACHE_DIR = OUTPUT_DIR / "cache"
OVERWRITE_OUTPUTS = False       # False permite reanudar sin repetir trabajo

LANDSAT_OUTPUT = OUTPUT_DIR / "dataset_landsat_sin_downscaling.csv"
SENTINEL3_OUTPUT = OUTPUT_DIR / "dataset_sentinel3_1km_sin_downscaling.csv"
PAIRED_OUTPUT = OUTPUT_DIR / "dataset_pares_landsat_sentinel3.csv"
NODES_OUTPUT = OUTPUT_DIR / "nodos_malla.csv"

print("Salidas:", OUTPUT_DIR.resolve())


## 2. CREDENCIALES — edita aquí o usa variables de entorno

**Propósito científico.** Este bloque obtiene credenciales exclusivamente del entorno local. Los secretos habilitan los servicios externos, pero no forman parte de los datos científicos ni deben persistirse en resultados.


In [ ]:
import os

# Los valores escritos aquí son fáciles de localizar. Para no guardar secretos
# en el notebook, es preferible definir las variables de entorno indicadas.
CREDENTIALS = {
    # Proyecto de Google Cloud registrado para Earth Engine.
    "earth_engine_project": os.getenv("EE_PROJECT", "tfm-gnn"),

    # Cliente OAuth creado en User settings > OAuth clients de Sentinel Hub.
    "sh_client_id": os.getenv("SH_CLIENT_ID", ""),
    "sh_client_secret": os.getenv("SH_CLIENT_SECRET", ""),

    # Token opcional: solo se usa si falla ERA5-Land en Earth Engine.
    "cds_api_key": os.getenv("CDS_API_KEY", ""),
}

CDS_API_URL = "https://cds.climate.copernicus.eu/api"
SH_TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
SH_CATALOGUE_URL = "https://sh.dataspace.copernicus.eu/catalog/v1/search"

def credential_is_placeholder(value):
    return (not value) or str(value).startswith("PON_AQUI")

print("EE project:", CREDENTIALS["earth_engine_project"])
print("Sentinel Hub configurado:", not credential_is_placeholder(CREDENTIALS["sh_client_id"]))
ERA5_SOURCE = os.getenv("ERA5_SOURCE", ERA5_SOURCE).strip().lower()
print("Fuente ERA5-Land preferida:", ERA5_SOURCE)
print("CDS fallback configurado:", not credential_is_placeholder(CREDENTIALS["cds_api_key"]))


## 3. Imports, validación y fechas efectivas

**Propósito científico.** Se cargan las librerías, se validan los parámetros y se construyen las fechas efectivas. Esta etapa fija las invariantes antes de cualquier consulta remota.


In [ ]:
import gzip
import io
import json
import math
import shutil
import time
import warnings
import zipfile
from collections import defaultdict
from datetime import timedelta
from pathlib import Path
from urllib.parse import quote

import cdsapi
import ee
import folium
import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
import rasterio
import requests
import xarray as xr
from pyproj import Transformer
from rasterstats import zonal_stats
from shapely.affinity import rotate as rotate_geometry
from shapely.geometry import Point, Polygon, box
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)

if NODE_SPACING_M <= 0 or HALF_SIDE_M <= 0:
    raise ValueError("NODE_SPACING_M y HALF_SIDE_M deben ser positivos.")
if ERA5_SOURCE not in {"earth_engine", "cds"}:
    raise ValueError("ERA5_SOURCE debe ser 'earth_engine' o 'cds'.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
(CACHE_DIR / "landsat").mkdir(exist_ok=True)
(CACHE_DIR / "sentinel3").mkdir(exist_ok=True)
(CACHE_DIR / "sentinel2").mkdir(exist_ok=True)
(CACHE_DIR / "era5").mkdir(exist_ok=True)
(CACHE_DIR / "cnig").mkdir(exist_ok=True)

normalized_date_windows = [
    (pd.Timestamp(start, tz="UTC"), pd.Timestamp(end, tz="UTC") + pd.Timedelta(days=1))
    for start, end in DATE_WINDOWS
]
if not normalized_date_windows or any(start >= end for start, end in normalized_date_windows):
    raise ValueError("DATE_WINDOWS debe contener al menos una ventana válida.")
query_start = min(start for start, _ in normalized_date_windows)
query_end = max(end for _, end in normalized_date_windows)

print(f"Rango global: {query_start.date()} — {(query_end - pd.Timedelta(days=1)).date()}")
print("Ventanas exactas:", [(start.date(), (end - pd.Timedelta(days=1)).date()) for start, end in normalized_date_windows])

## 4. ROI y malla métrica estable

**Propósito científico.** El ROI se transforma a una proyección métrica estable y se discretiza en nodos. Trabajar en EPSG:25830 permite expresar distancias y áreas de forma coherente para Sevilla.


In [ ]:
to_projected = Transformer.from_crs("EPSG:4326", PROJECTED_CRS, always_xy=True)
to_wgs84 = Transformer.from_crs(PROJECTED_CRS, "EPSG:4326", always_xy=True)
center_x, center_y = to_projected.transform(CENTER_LON, CENTER_LAT)
base_xmin, base_ymin = center_x - HALF_SIDE_M, center_y - HALF_SIDE_M
base_xmax, base_ymax = center_x + HALF_SIDE_M, center_y + HALF_SIDE_M
roi_projected = rotate_geometry(
    box(base_xmin, base_ymin, base_xmax, base_ymax),
    ROI_ROTATION_DEG, origin=(center_x, center_y), use_radians=False,
)
xmin, ymin, xmax, ymax = roi_projected.bounds
roi_wgs84 = gpd.GeoSeries([roi_projected], crs=PROJECTED_CRS).to_crs("EPSG:4326").iloc[0]
min_lon, min_lat, max_lon, max_lat = roi_wgs84.bounds

def load_or_create_nodes(nodes_csv=None):
    if nodes_csv:
        df = pd.read_csv(nodes_csv)
        lat_col = next((c for c in ["lat", "latitude"] if c in df.columns), None)
        lon_col = next((c for c in ["lon", "longitude"] if c in df.columns), None)
        if "node_id" not in df or not lat_col or not lon_col:
            raise ValueError("NODES_CSV necesita node_id y lat/lon o latitude/longitude.")
        keep = ["node_id", lat_col, lon_col] + [c for c in ["row", "col"] if c in df.columns]
        nodes = df[keep].drop_duplicates("node_id").rename(
            columns={lat_col: "latitude", lon_col: "longitude"}
        )
        nodes["x"], nodes["y"] = to_projected.transform(nodes.longitude, nodes.latitude)
        if not {"row", "col"}.issubset(df.columns):
            nodes["row"] = nodes["y"].rank(method="dense", ascending=False).astype(int) - 1
            nodes["col"] = nodes["x"].rank(method="dense").astype(int) - 1
    else:
        xs = np.arange(base_xmin + NODE_SPACING_M / 2, base_xmax, NODE_SPACING_M)
        ys = np.arange(base_ymax - NODE_SPACING_M / 2, base_ymin, -NODE_SPACING_M)
        angle = np.deg2rad(ROI_ROTATION_DEG)
        cos_angle, sin_angle = np.cos(angle), np.sin(angle)
        rows = []
        for row, y in enumerate(ys):
            for col, x in enumerate(xs):
                dx, dy = x - center_x, y - center_y
                rotated_x = center_x + dx * cos_angle - dy * sin_angle
                rotated_y = center_y + dx * sin_angle + dy * cos_angle
                lon, lat = to_wgs84.transform(rotated_x, rotated_y)
                rows.append({"node_id": f"r{row:03d}_c{col:03d}", "row": row, "col": col,
                             "x": rotated_x, "y": rotated_y, "latitude": lat, "longitude": lon})
        nodes = pd.DataFrame(rows)
    nodes = nodes.sort_values(["row", "col"]).reset_index(drop=True)
    nodes["row_norm"] = nodes["row"] / max(nodes["row"].max(), 1)
    nodes["col_norm"] = nodes["col"] / max(nodes["col"].max(), 1)
    return nodes

nodes = load_or_create_nodes(NODES_CSV)
nodes.to_csv(NODES_OUTPUT, index=False)
print(f"ROI: {2*HALF_SIDE_M:.0f} x {2*HALF_SIDE_M:.0f} m")
print(f"Malla: {nodes.row.nunique()} x {nodes.col.nunique()} = {len(nodes)} nodos; paso {NODE_SPACING_M} m")
display(nodes.head())

m = folium.Map(location=[CENTER_LAT, CENTER_LON], zoom_start=15, tiles="CartoDB positron")
folium.GeoJson(roi_wgs84.__geo_interface__, name="ROI", style_function=lambda _: {"color": "red"}).add_to(m)
for r in nodes.itertuples():
    folium.CircleMarker([r.latitude, r.longitude], radius=2, tooltip=str(r.node_id)).add_to(m)

import webbrowser
import os

m.save("mapa_roi.html")
webbrowser.open("file://" + os.path.abspath("mapa_roi.html"))

## 5. Autenticación y comprobación de servicios

**Propósito científico.** Se autentican los servicios de datos y se comprueba su disponibilidad. La autenticación es infraestructura; no modifica los algoritmos científicos siguientes.


In [ ]:
def init_earth_engine(project):
    try:
        ee.Initialize(project=project)
    except Exception:
        print("Earth Engine necesita autorización; se abrirá el flujo local en el navegador.")
        ee.Authenticate()
        ee.Initialize(project=project)
    print("✓ Earth Engine inicializado")

def get_sh_token():
    if credential_is_placeholder(CREDENTIALS["sh_client_id"]) or credential_is_placeholder(CREDENTIALS["sh_client_secret"]):
        raise ValueError("Completa SH_CLIENT_ID y SH_CLIENT_SECRET en CREDENTIALS.")
    response = requests.post(SH_TOKEN_URL, data={
        "grant_type": "client_credentials",
        "client_id": CREDENTIALS["sh_client_id"],
        "client_secret": CREDENTIALS["sh_client_secret"],
    }, timeout=60)
    response.raise_for_status()
    return response.json()["access_token"]

def get_cds_client():
    if credential_is_placeholder(CREDENTIALS["cds_api_key"]):
        raise ValueError("Completa cds_api_key en CREDENTIALS.")
    return cdsapi.Client(url=CDS_API_URL, key=CREDENTIALS["cds_api_key"], quiet=False)

init_earth_engine(CREDENTIALS["earth_engine_project"])
sh_token = get_sh_token()
cds_client = get_cds_client() if ERA5_SOURCE == "cds" else None
print("✓ Sentinel Hub autenticado")
print("✓ ERA5-Land usará Earth Engine" if ERA5_SOURCE == "earth_engine" else "✓ Cliente CDS preparado")


## 6. Funciones Earth Engine: Landsat, Sentinel-2 y variables estáticas

**Propósito científico.** Este bloque aplica la siguiente etapa lógica del pipeline científico existente. Sus entradas proceden de los bloques anteriores y sus resultados alimentan la armonización o salida posterior.


In [ ]:
from tools.era5_land import download_era5_land_gee, expanded_era5_days
from tools.scientific_variables import (
    add_landsat_variables as add_landsat_variable_bands,
    add_s2_variables,
    landsat_band_names,
    s2_band_names,
)

def nodes_to_ee_feature_collection(nodes_df):
    features = []
    for r in nodes_df.itertuples():
        props = {"node_id": str(r.node_id), "row": int(r.row), "col": int(r.col),
                 "latitude": float(r.latitude), "longitude": float(r.longitude)}
        features.append(ee.Feature(ee.Geometry.Point([r.longitude, r.latitude]), props))
    return ee.FeatureCollection(features)

nodes_fc = nodes_to_ee_feature_collection(nodes)
roi_ee = ee.Geometry.Polygon(list(roi_wgs84.exterior.coords))

def mask_landsat_l2(img):
    qa = img.select("QA_PIXEL")
    clear = (qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
             .And(qa.bitwiseAnd(1 << 5).eq(0)).And(qa.bitwiseAnd(1 << 7).eq(0)))
    return img.updateMask(clear)

def add_landsat_variables(img):
    return add_landsat_variable_bands(img, REQUESTED_VARIABLES)

def landsat_collection(start, end):
    base = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").merge(
        ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
    ).filterBounds(roi_ee).filter(
        ee.Filter.calendarRange(min(ALLOWED_MONTHS), max(ALLOWED_MONTHS), "month")
    ).filter(ee.Filter.lte("CLOUD_COVER", MAX_LANDSAT_CLOUD))
    selected = None
    for window_start, window_end in normalized_date_windows:
        subset = base.filterDate(window_start.strftime("%Y-%m-%d"), window_end.strftime("%Y-%m-%d"))
        selected = subset if selected is None else selected.merge(subset)
    return selected.sort("system:time_start").map(mask_landsat_l2).map(add_landsat_variables)

def select_test_image(collection, anchor_time):
    target_ms = int(anchor_time.timestamp() * 1000)
    return ee.Image(collection.map(lambda img: img.set(
        "distance_to_test", ee.Number(img.get("system:time_start")).subtract(target_ms).abs()
    )).sort("distance_to_test").first())

def fc_to_dataframe(fc):
    info = fc.getInfo()
    rows = []
    for feature in info.get("features", []):
        props = feature.get("properties", {}).copy()
        geom = feature.get("geometry")
        if geom and geom.get("type") == "Point":
            props.setdefault("longitude", geom["coordinates"][0])
            props.setdefault("latitude", geom["coordinates"][1])
        rows.append(props)
    return pd.DataFrame(rows)

def sample_image_at_nodes(image, bands, scale):
    reduced = image.select(bands).reduceRegions(nodes_fc, ee.Reducer.first(), scale=scale)
    df = fc_to_dataframe(reduced)
    return df.rename(columns={f"first": bands[0]}) if len(bands) == 1 and "first" in df else df

def mask_sentinel2(img):
    scl = img.select("SCL")
    valid = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    return img.updateMask(valid)

def sentinel2_predictors_nearest(acquisition_time):
    start = (acquisition_time - pd.Timedelta(days=S2_MAX_DAY_DIFFERENCE)).strftime("%Y-%m-%d")
    end = (acquisition_time + pd.Timedelta(days=S2_MAX_DAY_DIFFERENCE + 1)).strftime("%Y-%m-%d")
    target_ms = int(acquisition_time.timestamp() * 1000)
    def add_s2_predictors(img):
        return add_s2_variables(img, REQUESTED_VARIABLES)

    collection = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi_ee)
                  .filterDate(start, end).filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", MAX_SENTINEL2_CLOUD))
                  .map(mask_sentinel2).map(add_s2_predictors))
    count = collection.size().getInfo()
    if count == 0:
        return pd.DataFrame(), None
    image = select_test_image(collection, acquisition_time)
    ts = pd.to_datetime(image.get("system:time_start").getInfo(), unit="ms", utc=True)
    df = sample_image_at_nodes(image, s2_band_names(REQUESTED_VARIABLES), NODE_SPACING_M)
    return df, ts

node_hash = int(pd.util.hash_pandas_object(
    nodes[["node_id", "latitude", "longitude"]].astype({"node_id": str}), index=False
).sum())
STATIC_CACHE_KEY = (
    f"{CENTER_LAT:.5f}_{CENTER_LON:.5f}_{HALF_SIDE_M:.0f}_{ROI_ROTATION_DEG:.2f}_"
    f"{NODE_SPACING_M:.0f}_{int(INCLUDE_URBAN_MORPHOLOGY)}_{node_hash}"
).replace("-", "m").replace(".", "p")
STATIC_CACHE_FILE = CACHE_DIR / "static" / f"node_static_{STATIC_CACHE_KEY}.csv"
STATIC_CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)

STATIC_CACHE_HIT = False
if STATIC_CACHE_FILE.exists() and not OVERWRITE_OUTPUTS:
    cached_static = pd.read_csv(STATIC_CACHE_FILE)
    expected_ids = set(nodes["node_id"].astype(str))
    cached_ids = set(cached_static.get("node_id", pd.Series(dtype=str)).astype(str))
    required_static = {"node_id", "DEM", "aspect_ratio", "has_buildings"} | ({"sky_view_factor"} if "sky_view_factor" in REQUESTED_VARIABLES else set())
    if expected_ids == cached_ids and required_static.issubset(cached_static.columns):
        static_df = cached_static
        static_df["node_id"] = static_df["node_id"].astype(str)
        STATIC_CACHE_HIT = True
        print("✓ Variables estáticas reutilizadas desde:", STATIC_CACHE_FILE)

if not STATIC_CACHE_HIT:
    dem_image = ee.Image("USGS/SRTMGL1_003").select("elevation").rename("DEM")
    dem_sample = sample_image_at_nodes(dem_image, ["DEM"], NODE_SPACING_M)
    if "first" in dem_sample.columns:
        dem_sample = dem_sample.rename(columns={"first": "DEM"})
    nodes["node_id"] = nodes["node_id"].astype(str)
    dem_sample["node_id"] = dem_sample["node_id"].astype(str)
    static_df = nodes.merge(
        dem_sample[[c for c in ["node_id", "DEM"] if c in dem_sample.columns]],
        on="node_id", how="left"
    )
    print("Variables estáticas GEE calculadas:", static_df.shape)

display(static_df.head())


## 7. Morfología urbana opcional: CNIG + OpenStreetMap

**Propósito científico.** Este bloque aplica la siguiente etapa lógica del pipeline científico existente. Sus entradas proceden de los bloques anteriores y sus resultados alimentan la armonización o salida posterior.


In [ ]:
from tools.urban_morphology import approximate_sky_view_factor

def build_node_cells(nodes_df):
    geoms = [box(r.x - NODE_SPACING_M/2, r.y - NODE_SPACING_M/2,
                 r.x + NODE_SPACING_M/2, r.y + NODE_SPACING_M/2) for r in nodes_df.itertuples()]
    return gpd.GeoDataFrame(nodes_df[["node_id", "row", "col"]].copy(), geometry=geoms, crs=PROJECTED_CRS)

def download_cnig_building_height(target):
    if target.exists() and not OVERWRITE_OUTPUTS:
        return target
    params = [("service", "WCS"), ("version", "2.0.1"), ("request", "GetCoverage"),
              ("coverageId", "mdsn_e025"), ("format", "image/tiff"),
              ("subset", f"x({xmin-250},{xmax+250})"), ("subset", f"y({ymin-250},{ymax+250})")]
    response = requests.get("https://wcs-mds.idee.es/mds", params=params, timeout=180)
    response.raise_for_status()
    if b"<?xml" in response.content[:1000]:
        raise RuntimeError(response.text[:2000])
    target.write_bytes(response.content)
    return target

def compute_urban_morphology(nodes_df):
    result = nodes_df[["node_id"]].copy()
    result["building_height_m"] = 0.0
    result["bld_footprint_m2"] = 0.0
    result["has_buildings"] = 0
    result["aspect_ratio"] = 0.0
    result["sky_view_factor"] = 1.0
    if not INCLUDE_URBAN_MORPHOLOGY:
        return result

    cells = build_node_cells(nodes_df)
    tif = download_cnig_building_height(CACHE_DIR / "cnig" / f"building_height_cnig_{STATIC_CACHE_KEY}.tif")
    if "sky_view_factor" in REQUESTED_VARIABLES:
        result["sky_view_factor"] = approximate_sky_view_factor(nodes_df, tif, node_crs=PROJECTED_CRS)
    with rasterio.open(tif) as src:
        arr = src.read(1).astype(float)
        if src.nodata is not None:
            arr[np.isclose(arr, src.nodata)] = np.nan
        arr[arr < 2.0] = np.nan
        means = zonal_stats(cells.to_crs(src.crs).geometry, arr, affine=src.transform,
                            stats=["mean"], nodata=np.nan, all_touched=True)
    result["building_height_m"] = [x["mean"] if x["mean"] is not None else 0.0 for x in means]

    print("Descargando huellas de edificios OSM...")
    buildings = ox.features_from_polygon(roi_wgs84.buffer(0.001), tags={"building": True})
    if len(buildings):
        buildings = buildings.reset_index(drop=True)
        buildings = buildings[buildings.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
        buildings = gpd.GeoDataFrame(buildings, geometry="geometry", crs="EPSG:4326").to_crs(PROJECTED_CRS)
        footprints = []
        counts = []
        for cell in cells.geometry:
            hits = buildings[buildings.intersects(cell)]
            intersections = hits.geometry.intersection(cell)
            footprints.append(float(intersections.area.sum()))
            counts.append(int((intersections.area > 0).sum()))
        result["bld_footprint_m2"] = footprints
        result["has_buildings"] = (np.array(counts) > 0).astype(int)
        free_area = np.maximum(NODE_SPACING_M**2 - result["bld_footprint_m2"], 0)
        street_width = np.where(np.array(counts) > 0, np.sqrt(free_area / np.maximum(counts, 1)), np.nan)
        result["aspect_ratio"] = (result["building_height_m"] / street_width).replace([np.inf, -np.inf], np.nan).fillna(0)
    return result

if STATIC_CACHE_HIT:
    print("✓ DEM y morfología urbana ya disponibles; no se repiten descargas estáticas.")
else:
    urban_df = compute_urban_morphology(nodes)
    static_df = static_df.merge(urban_df, on="node_id", how="left")
    static_df.to_csv(STATIC_CACHE_FILE, index=False)
    print("✓ Variables estáticas guardadas para reutilización:", STATIC_CACHE_FILE)
display(static_df.head())

## 8. Landsat procesado como una tabla única en Earth Engine

Las escenas conservan las mismas máscaras, bandas, escala de 30 m y nodos. La diferencia es operativa: Earth Engine compone una sola `FeatureCollection` y `computeFeatures` la transfiere paginada directamente a un `DataFrame`. Se evita una llamada síncrona por escena y no se utiliza Google Drive.


**Propósito científico.** Este bloque aplica la siguiente etapa lógica del pipeline científico existente. Sus entradas proceden de los bloques anteriores y sus resultados alimentan la armonización o salida posterior.


In [ ]:
def get_landsat_collection():
    collection = landsat_collection(query_start.strftime("%Y-%m-%d"), query_end.strftime("%Y-%m-%d"))
    scene_count = int(collection.size().getInfo())
    if scene_count == 0:
        raise RuntimeError("No hay imágenes Landsat en la ventana configurada.")
    return collection, scene_count

def landsat_feature_collection(collection, bands):
    def sample_scene(image):
        time_start = ee.Number(image.get("system:time_start"))
        scene_id = ee.String(image.get("system:index"))
        date = ee.Date(time_start).format("YYYY-MM-dd")
        samples = image.select(bands).reduceRegions(
            collection=nodes_fc,
            reducer=ee.Reducer.first(),
            scale=30,
            tileScale=4,
        )
        return samples.map(lambda feature: feature.set({
            "scene_id": scene_id,
            "time_start": time_start,
            "date": date,
        }))

    return ee.FeatureCollection(collection.map(sample_scene).flatten())

def landsat_batch_cache_file(bands):
    import hashlib
    signature = {
        "version": 2,
        "date_windows": [(str(start), str(end)) for start, end in DATE_WINDOWS],
        "bands": list(bands),
        "max_cloud": MAX_LANDSAT_CLOUD,
        "node_hash": node_hash,
    }
    digest = hashlib.sha1(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:16]
    return CACHE_DIR / "landsat" / f"landsat_batch_{digest}.csv"

def download_landsat_rows():
    collection, scene_count = get_landsat_collection()
    bands = landsat_band_names(REQUESTED_VARIABLES)
    cache_file = landsat_batch_cache_file(bands)
    error_file = OUTPUT_DIR / "errores_landsat.csv"

    if cache_file.exists() and not OVERWRITE_OUTPUTS:
        frame = pd.read_csv(cache_file)
        print("✓ Landsat reutilizado desde la caché tabular")
    else:
        print(
            f"Procesando {scene_count} escenas Landsat y {len(nodes)} nodos "
            "en una operación tabular de Earth Engine..."
        )
        try:
            samples = landsat_feature_collection(collection, bands)
            frame = ee.data.computeFeatures({
                "expression": samples,
                "fileFormat": "PANDAS_DATAFRAME",
                "pageSize": 5000,
                "workloadTag": "lst-tfm-landsat-batch",
            })
            if not isinstance(frame, pd.DataFrame) or frame.empty:
                raise RuntimeError("Earth Engine no devolvió filas en el lote Landsat.")
            temporary = cache_file.with_suffix(cache_file.suffix + ".part")
            frame.to_csv(temporary, index=False)
            temporary.replace(cache_file)
            error_file.unlink(missing_ok=True)
        except Exception as exc:
            cache_file.with_suffix(cache_file.suffix + ".part").unlink(missing_ok=True)
            pd.DataFrame([{"batch": "landsat", "error": str(exc)}]).to_csv(error_file, index=False)
            raise

    required = {"node_id", "scene_id", "time_start", "date", *bands}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise RuntimeError(f"El lote Landsat no contiene las columnas esperadas: {missing}")
    frame["node_id"] = frame["node_id"].astype(str)
    frame["time_start"] = pd.to_numeric(frame["time_start"], errors="raise")

    base_cols = ["node_id", "row", "col", "row_norm", "col_norm", "latitude", "longitude"]
    node_table = nodes[base_cols].copy()
    node_table["node_id"] = node_table["node_id"].astype(str)
    out = frame.drop(
        columns=[c for c in [*base_cols[1:], "geo", ".geo"] if c in frame], errors="ignore"
    ).merge(node_table, on="node_id", how="right")
    out = out.merge(
        static_df.drop(
            columns=["row", "col", "row_norm", "col_norm", "latitude", "longitude", "x", "y"],
            errors="ignore",
        ),
        on="node_id",
        how="left",
    )
    return out

landsat_df = download_landsat_rows()
landsat_df["acquisition_dt_utc"] = pd.to_datetime(landsat_df["time_start"], unit="ms", utc=True)
print("Landsat:", landsat_df.shape, "| escenas:", landsat_df.scene_id.nunique())
display(landsat_df.head())


## 9. ERA5-Land: planificación compartida, caché y variables derivadas

**Propósito científico.** Este bloque aplica la siguiente etapa lógica del pipeline científico existente. Sus entradas proceden de los bloques anteriores y sus resultados alimentan la armonización o salida posterior.


In [ ]:
from tools.scientific_variables import (
    deaccumulate_era5_land,
    derive_era5,
    era5_accumulated_short_names,
    era5_output_columns,
    era5_request_variables,
    era5_short_names,
)

ERA5_VARIABLES = era5_request_variables(REQUESTED_VARIABLES)
ERA5_SHORT_VARIABLES = era5_short_names(REQUESTED_VARIABLES)
ERA5_ACCUMULATED_COLUMNS = era5_accumulated_short_names(REQUESTED_VARIABLES)
ERA5_AREA = [max_lat + 0.15, min_lon - 0.15, min_lat - 0.15, max_lon + 0.15]  # N,W,S,E

def fix_zip_disguised_as_nc(path):
    if not zipfile.is_zipfile(path):
        return
    with zipfile.ZipFile(path) as zf:
        nc_names = [n for n in zf.namelist() if n.lower().endswith(".nc")]
        if not nc_names:
            raise RuntimeError(f"{path.name} es ZIP pero no contiene NetCDF.")
        with zf.open(nc_names[0]) as src, open(str(path) + ".tmp", "wb") as dst:
            shutil.copyfileobj(src, dst)
    Path(str(path) + ".tmp").replace(path)

def era5_month_cache_file(year, month, month_days):
    import hashlib
    signature = {
        "version": 3,
        "variables": ERA5_VARIABLES,
        "area": [round(value, 6) for value in ERA5_AREA],
        "year": year,
        "month": month,
        "days": sorted({int(day.day) for day in month_days}),
    }
    digest = hashlib.sha1(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:16]
    return CACHE_DIR / "era5" / f"era5_shared_{year}_{month:02d}_{digest}.nc"

def download_shared_era5_cds(target_times):
    client = cds_client or get_cds_client()
    days = expanded_era5_days(target_times)
    groups = defaultdict(list)
    for day in days:
        groups[(day.year, day.month)].append(day)
    paths = []
    for (year, month), month_days in tqdm(sorted(groups.items()), desc="ERA5 compartido"):
        target = era5_month_cache_file(year, month, month_days)
        if not target.exists() or OVERWRITE_OUTPUTS:
            request = {
                "variable": ERA5_VARIABLES,
                "year": str(year),
                "month": f"{month:02d}",
                "day": [f"{day.day:02d}" for day in sorted(set(month_days))],
                "time": [f"{hour:02d}:00" for hour in range(24)],
                "area": ERA5_AREA,
                "data_format": "netcdf",
                "download_format": "unarchived",
            }
            partial = target.with_suffix(target.suffix + ".part")
            partial.unlink(missing_ok=True)
            client.retrieve("reanalysis-era5-land", request, str(partial))
            partial.replace(target)
        fix_zip_disguised_as_nc(target)
        paths.append(target)
    return paths

def read_shared_era5_cds(paths):
    frames = []
    for path in paths:
        ds = xr.open_dataset(path)
        available = [name for name in ERA5_SHORT_VARIABLES if name in ds.variables]
        missing = [name for name in ERA5_SHORT_VARIABLES if name not in ds.variables]
        if missing:
            raise RuntimeError(f"ERA5-Land no devolvió las variables esperadas: {missing}")
        frame = ds[available].to_dataframe().reset_index()
        time_col = next(c for c in ["valid_time", "time", "datetime", "date"] if c in frame)
        frame = frame.rename(columns={time_col: "era5_dt_utc"})
        frame["era5_dt_utc"] = pd.to_datetime(frame["era5_dt_utc"], utc=True)
        frames.append(frame)
        ds.close()
    regional = pd.concat(frames, ignore_index=True)
    regional = regional.groupby("era5_dt_utc", as_index=False)[ERA5_SHORT_VARIABLES].mean()
    regional = deaccumulate_era5_land(regional, ERA5_ACCUMULATED_COLUMNS)
    return derive_era5(regional)

def load_shared_era5(target_times):
    if ERA5_SOURCE == "earth_engine":
        try:
            regional, info = download_era5_land_gee(
                target_times, area_nwse=ERA5_AREA, selected_variables=REQUESTED_VARIABLES,
                cache_dir=CACHE_DIR / "era5_gee", overwrite=OVERWRITE_OUTPUTS, ee_module=ee,
            )
            hourly, daily = derive_era5(regional)
            return hourly, daily, info
        except Exception as earth_engine_error:
            if credential_is_placeholder(CREDENTIALS["cds_api_key"]):
                raise RuntimeError(
                    "ERA5-Land falló en Earth Engine y CDS_API_KEY no está configurado como fallback."
                ) from earth_engine_error
            print("Earth Engine no completó ERA5-Land; se usará el fallback CDS.")
    paths = download_shared_era5_cds(target_times)
    hourly, daily = read_shared_era5_cds(paths)
    info = {
        "source": "cds", "requested_days": len(expanded_era5_days(target_times)),
        "downloaded_hours": int(hourly["era5_dt_utc"].nunique()), "cache_files": len(paths),
    }
    return hourly, daily, info

def attach_era5_tables(data, datetime_col, hourly, daily):
    out = data.copy()
    out["era5_dt_utc"] = pd.to_datetime(out[datetime_col], utc=True).dt.round("h")
    hourly_columns = [c for c in era5_output_columns(REQUESTED_VARIABLES) if c in hourly.columns]
    out = out.merge(hourly[["era5_dt_utc", *hourly_columns]], on="era5_dt_utc", how="left")
    out["day"] = pd.to_datetime(out[datetime_col], utc=True).dt.floor("D")
    daily_columns = [c for c in ["rain_1d", "rain_3d", "rain_3d_log", "is_rainy"] if c in daily.columns]
    return out.merge(daily[["day", *daily_columns]], on="day", how="left").drop(columns="day")

## 10. Variables temporales comunes

**Propósito científico.** Este bloque aplica la siguiente etapa lógica del pipeline científico existente. Sus entradas proceden de los bloques anteriores y sus resultados alimentan la armonización o salida posterior.


In [ ]:
def add_temporal_features(df, datetime_col):
    out = df.copy()
    dt = pd.to_datetime(out[datetime_col], utc=True)
    out["date"] = dt.dt.strftime("%Y-%m-%d")
    out["day_of_year"] = dt.dt.dayofyear
    out["sin_doy"] = np.sin(2*np.pi*out.day_of_year/365.25)
    out["cos_doy"] = np.cos(2*np.pi*out.day_of_year/365.25)
    return out

## 11. Sentinel-3 SLSTR L2 LST diario mediante lotes estadísticos

Sentinel Hub agrupa intervalos diarios y devuelve estadísticas por cada celda de 1 km. Se conserva la escena diaria más cercana a la hora Landsat mediana, se rellenan días intermedios y no se descargan productos completos.


**Propósito científico.** Este bloque aplica la siguiente etapa lógica del pipeline científico existente. Sus entradas proceden de los bloques anteriores y sus resultados alimentan la armonización o salida posterior.


In [ ]:
from tools.sentinelhub_s3 import (
    S3_COLLECTION,
    catalogue_rows,
    choose_s3_products,
    landsat_reference_hour,
)

def refresh_sh_token():
    global sh_token
    sh_token = get_sh_token()

def sh_post(url, payload, timeout=180):
    global sh_token
    last_response = None
    for attempt in range(5):
        response = requests.post(
            url,
            headers={"Authorization": f"Bearer {sh_token}"},
            json=payload,
            timeout=timeout,
        )
        last_response = response
        if response.status_code == 401:
            response.close()
            refresh_sh_token()
            continue
        if response.status_code == 429 or response.status_code >= 500:
            retry_after = response.headers.get("Retry-After")
            wait_seconds = float(retry_after) if retry_after and retry_after.isdigit() else 2 ** attempt
            response.close()
            time.sleep(min(wait_seconds, 30))
            continue
        if response.status_code >= 400:
            excerpt = response.text[:1200]
            status = response.status_code
            response.close()
            raise RuntimeError(f"Sentinel Hub HTTP {status}: {excerpt}")
        return response
    assert last_response is not None
    status = last_response.status_code
    excerpt = last_response.text[:1200]
    last_response.close()
    raise RuntimeError(f"Sentinel Hub agotó los reintentos (HTTP {status}): {excerpt}")

def search_s3_products(date_windows):
    features = []
    for start, end in date_windows:
        if pd.Timestamp(end).year < 2016:
            continue
        base_payload = {
            "collections": [S3_COLLECTION],
            "bbox": [min_lon, min_lat, max_lon, max_lat],
            "datetime": f"{start.strftime('%Y-%m-%dT%H:%M:%SZ')}/{end.strftime('%Y-%m-%dT%H:%M:%SZ')}",
            "limit": 100,
        }
        current_payload = base_payload.copy()
        while True:
            response = sh_post(SH_CATALOGUE_URL, current_payload, timeout=120)
            payload = response.json()
            response.close()
            features.extend(payload.get("features", []))
            next_body = next(
                (link.get("body") for link in payload.get("links", []) if link.get("rel") == "next"),
                None,
            )
            if not next_body:
                break
            current_payload = base_payload | next_body

    products = catalogue_rows(features)
    return choose_s3_products(
        products,
        morning_start_hour=S3_MORNING_START_UTC,
        morning_end_hour=S3_MORNING_END_UTC,
        target_hour=s3_target_hour_utc,
        reference_datetimes=None,
        tolerance_days=S3_REFERENCE_TOLERANCE_DAYS,
    )

s3_target_hour_utc = landsat_reference_hour(landsat_df["acquisition_dt_utc"], S3_TARGET_HOUR_UTC)
s3_products, s3_daily_count = search_s3_products(normalized_date_windows)
print(f"Hora Landsat de referencia: {s3_target_hour_utc:.2f} UTC")
print(f"Adquisiciones Sentinel-3 diarias seleccionadas: {len(s3_products)}")
print("Se usará una adquisición diaria disponible, siempre la más próxima a la hora Landsat.")
display(s3_products[["Name", "acquisition_dt_utc", "cloud_cover"]].head())


**Propósito científico.** Este bloque aplica la siguiente etapa lógica del pipeline científico existente. Sus entradas proceden de los bloques anteriores y sus resultados alimentan la armonización o salida posterior.


In [ ]:
from tools.sentinelhub_s3 import (
    S3_PROCESS_OUTPUT_COLUMNS,
    S3_STATISTICS_URL,
    aligned_grid_cells,
    aligned_output_grid,
    statistics_request_payload,
    statistics_response_rows,
)

import hashlib

s3_roi_request = gpd.GeoSeries([roi_projected], crs=PROJECTED_CRS).to_crs(S3_REQUEST_CRS).iloc[0]
s3_to_wgs84 = Transformer.from_crs(S3_REQUEST_CRS, "EPSG:4326", always_xy=True)
s3_process_bounds, s3_width, s3_height = aligned_output_grid(
    s3_roi_request.bounds, SENTINEL3_NATIVE_M
)
s3_cells = aligned_grid_cells(s3_process_bounds, s3_width, s3_height)
s3_grid_signature = {
    "version": 4,
    "method": "statistical_api_daily",
    "bounds": [round(value, 3) for value in s3_process_bounds],
    "width": s3_width,
    "height": s3_height,
    "project_crs": PROJECTED_CRS,
    "request_crs": S3_REQUEST_CRS,
    "target_hour_utc": round(s3_target_hour_utc, 4),
    "max_uncertainty_k": S3_MAX_UNCERTAINTY_K,
    "products": s3_products["Id"].astype(str).tolist(),
}
S3_GRID_CACHE_KEY = hashlib.sha1(
    json.dumps(s3_grid_signature, sort_keys=True).encode()
).hexdigest()[:16]
S3_STATS_CACHE_DIR = CACHE_DIR / "sentinel3_stats"
S3_STATS_CACHE_DIR.mkdir(exist_ok=True)
S3_BATCH_CACHE = S3_STATS_CACHE_DIR / f"sentinel3_daily_{S3_GRID_CACHE_KEY}.csv.gz"
S3_QUALITY_CACHE = S3_STATS_CACHE_DIR / f"sentinel3_quality_{S3_GRID_CACHE_KEY}.json"
print(
    f"Malla Sentinel-3: {s3_width} x {s3_height} celdas de {SENTINEL3_NATIVE_M:g} m; "
    f"Statistical API evitará una petición por escena."
)

def utc_timestamp(value):
    timestamp = pd.Timestamp(value)
    return timestamp.tz_localize("UTC") if timestamp.tzinfo is None else timestamp.tz_convert("UTC")

def stats_task_cache(cell, start, end):
    signature = {
        "grid": S3_GRID_CACHE_KEY,
        "row": cell["row"],
        "col": cell["col"],
        "start": utc_timestamp(start).isoformat(),
        "end": utc_timestamp(end).isoformat(),
    }
    digest = hashlib.sha1(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:16]
    return S3_STATS_CACHE_DIR / f"stats_{digest}.json.gz"

def download_all_s3_pixels(products):
    if S3_BATCH_CACHE.exists() and not OVERWRITE_OUTPUTS:
        print("✓ Serie diaria Sentinel-3 reutilizada desde la caché estadística")
        frame = pd.read_csv(S3_BATCH_CACHE, parse_dates=["acquisition_dt_utc"])
        quality = json.loads(S3_QUALITY_CACHE.read_text(encoding="utf-8")) if S3_QUALITY_CACHE.exists() else {
            "catalog_dates": int(frame["acquisition_dt_utc"].dt.date.nunique()),
            "valid_dates": int(frame["acquisition_dt_utc"].dt.date.nunique()),
            "valid_cell_days": len(frame), "cache_reused": True,
        }
        quality["cache_reused"] = True
        return frame, quality

    product_lookup = products.copy()
    product_lookup["date_key"] = pd.to_datetime(product_lookup["date"]).dt.strftime("%Y-%m-%d")
    product_lookup = product_lookup.drop_duplicates("date_key").set_index("date_key")
    tasks = []
    for start, end in normalized_date_windows:
        start_utc = utc_timestamp(start).normalize()
        end_utc = utc_timestamp(end).normalize()
        product_dates = pd.to_datetime(products["acquisition_dt_utc"], utc=True).dt.normalize()
        available = products.loc[(product_dates >= start_utc) & (product_dates < end_utc)]
        if available.empty:
            continue
        request_start = pd.to_datetime(available["acquisition_dt_utc"], utc=True).min().normalize()
        request_end = pd.to_datetime(available["acquisition_dt_utc"], utc=True).max().normalize() + pd.Timedelta(days=1)
        for cell in s3_cells:
            tasks.append((cell, request_start, request_end))

    records, errors = [], []
    returned_dates, valid_dates, cloudy_dates = set(), set(), set()
    returned_cell_days = cloudy_cell_days = invalid_cell_days = 0
    consecutive_errors = 0
    for task_number, (cell, start, end) in enumerate(tqdm(tasks, desc="Sentinel-3 diario por lotes"), start=1):
        cache_path = stats_task_cache(cell, start, end)
        try:
            if cache_path.exists() and not OVERWRITE_OUTPUTS:
                with gzip.open(cache_path, "rt", encoding="utf-8") as source:
                    payload = json.load(source)
            else:
                request_payload = statistics_request_payload(
                    cell["bounds"], S3_REQUEST_CRS, start, end, s3_target_hour_utc,
                    resolution_m=SENTINEL3_NATIVE_M,
                    morning_start_hour=S3_MORNING_START_UTC,
                    morning_end_hour=S3_MORNING_END_UTC,
                )
                response = sh_post(S3_STATISTICS_URL, request_payload, timeout=180)
                payload = response.json()
                response.close()
                temporary = cache_path.with_suffix(cache_path.suffix + ".part")
                with gzip.open(temporary, "wt", encoding="utf-8") as destination:
                    json.dump(payload, destination, separators=(",", ":"))
                temporary.replace(cache_path)
            consecutive_errors = 0
            longitude, latitude = s3_to_wgs84.transform(cell["x"], cell["y"])
            for row in statistics_response_rows(payload):
                date_key = pd.Timestamp(row["date"]).strftime("%Y-%m-%d")
                if date_key not in product_lookup.index:
                    continue
                product = product_lookup.loc[date_key]
                returned_dates.add(date_key)
                returned_cell_days += 1
                lst_value = float(row.get("LST", np.nan))
                uncertainty = float(row.get("LST_uncertainty", np.nan))
                confidence = int(round(float(row.get("CONFIDENCE", 0))))
                if not np.isfinite(lst_value) or not 223 <= lst_value <= 360:
                    invalid_cell_days += 1
                    continue
                if not np.isfinite(uncertainty) or not 0 <= uncertainty <= S3_MAX_UNCERTAINTY_K:
                    invalid_cell_days += 1
                    continue
                if confidence & (1 << 14):
                    cloudy_cell_days += 1
                    cloudy_dates.add(date_key)
                    continue
                acquisition = pd.Timestamp(product["acquisition_dt_utc"])
                record = {
                    "s3_pixel_id": f"{acquisition:%Y%m%dT%H%M}_{cell['row']}_{cell['col']}",
                    "latitude": latitude,
                    "longitude": longitude,
                    "s3_row": int(cell["row"]),
                    "s3_col": int(cell["col"]),
                    "s3_product_id": str(product["Id"]),
                    "s3_product_name": str(product["Name"]),
                    "acquisition_dt_utc": acquisition,
                }
                valid_dates.add(date_key)
                record.update({target: row[source] for source, target in S3_PROCESS_OUTPUT_COLUMNS.items()})
                record["quality_flag_1km"] = confidence
                records.append(record)
        except Exception as exc:
            consecutive_errors += 1
            errors.append({
                "task": task_number,
                "row": cell["row"],
                "col": cell["col"],
                "start": str(start),
                "end": str(end),
                "error": str(exc),
            })
            pd.DataFrame(errors).to_csv(OUTPUT_DIR / "errores_sentinel3.csv", index=False)
            if consecutive_errors >= 3:
                raise RuntimeError(
                    "Sentinel-3 falló en tres lotes consecutivos. Ejecuta 04_diagnostico_sentinel3.ipynb."
                ) from exc

    if not records:
        raise RuntimeError("Statistical API no devolvió ningún píxel Sentinel-3 válido.")
    frame = pd.DataFrame(records).drop_duplicates(["acquisition_dt_utc", "s3_row", "s3_col"])
    catalog_dates = set(product_lookup.index.astype(str))
    expected_cell_days = len(catalog_dates) * len(s3_cells)
    quality = {
        "catalog_dates": len(catalog_dates), "returned_dates": len(returned_dates),
        "valid_dates": len(valid_dates), "cloudy_dates": len(cloudy_dates),
        "missing_dates": len(catalog_dates - returned_dates),
        "expected_cell_days": expected_cell_days, "returned_cell_days": returned_cell_days,
        "valid_cell_days": len(frame), "cloudy_cell_days": cloudy_cell_days,
        "invalid_cell_days": invalid_cell_days,
        "missing_cell_days": max(expected_cell_days - returned_cell_days, 0),
        "failed_batches": len(errors), "cache_reused": False,
    }
    temporary = S3_BATCH_CACHE.with_suffix(S3_BATCH_CACHE.suffix + ".part")
    frame.to_csv(temporary, index=False, compression="gzip")
    temporary.replace(S3_BATCH_CACHE)
    S3_QUALITY_CACHE.write_text(json.dumps(quality, indent=2), encoding="utf-8")
    return frame, quality

s3_pixels, s3_quality = download_all_s3_pixels(s3_products)
print("Píxeles Sentinel-3 válidos:", len(s3_pixels), "| fechas:", s3_pixels.acquisition_dt_utc.nunique())
display(s3_pixels.head())


## 12. Sentinel-2 tabular y agregación fina sobre cada píxel Sentinel-3

**Propósito científico.** Este bloque aplica la siguiente etapa lógica del pipeline científico existente. Sus entradas proceden de los bloques anteriores y sus resultados alimentan la armonización o salida posterior.


In [ ]:
from tools.scientific_variables import s2_band_names

def static_predictors_by_node():
    drop = ["row", "col", "row_norm", "col_norm", "latitude", "longitude", "x", "y"]
    predictors = static_df.drop(columns=[c for c in drop if c in static_df], errors="ignore")
    return nodes[["node_id", "x", "y"]].merge(predictors, on="node_id", how="left")

def sentinel2_batch_cache_file(acquisitions, bands):
    import hashlib
    signature = {
        "version": 3,
        "acquisitions": [pd.Timestamp(item).isoformat() for item in acquisitions],
        "bands": list(bands),
        "node_hash": node_hash,
        "max_cloud": MAX_SENTINEL2_CLOUD,
        "max_day_difference": S2_MAX_DAY_DIFFERENCE,
        "date_windows": [(start.isoformat(), end.isoformat()) for start, end in normalized_date_windows],
    }
    digest = hashlib.sha1(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:16]
    return CACHE_DIR / "sentinel2" / f"sentinel2_batch_{digest}.csv"

def sentinel2_source_collection(bands):
    filters = []
    for start, end in normalized_date_windows:
        extended_start = (start - pd.Timedelta(days=S2_MAX_DAY_DIFFERENCE)).strftime("%Y-%m-%d")
        extended_end = (end + pd.Timedelta(days=S2_MAX_DAY_DIFFERENCE)).strftime("%Y-%m-%d")
        filters.append(ee.Filter.date(extended_start, extended_end))
    date_filter = filters[0]
    for item in filters[1:]:
        date_filter = ee.Filter.Or(date_filter, item)
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(roi_ee)
        .filter(date_filter)
        .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", MAX_SENTINEL2_CLOUD))
        .map(mask_sentinel2)
        .map(lambda image: add_s2_variables(image, REQUESTED_VARIABLES))
        .select(bands)
    )

def sentinel2_feature_collection(bands):
    collection = sentinel2_source_collection(bands)
    def sample_scene(image):
        return image.reduceRegions(
            collection=nodes_fc,
            reducer=ee.Reducer.first(),
            scale=NODE_SPACING_M,
            tileScale=4,
        ).map(lambda feature: feature.set({
            "s2_source_time_start": image.get("system:time_start"),
        }))
    return collection.map(sample_scene).flatten()

def download_sentinel2_batch(acquisitions, bands):
    acquisitions = pd.DatetimeIndex(pd.to_datetime(acquisitions, utc=True)).sort_values().unique()
    cache_file = sentinel2_batch_cache_file(acquisitions, bands)
    if cache_file.exists() and not OVERWRITE_OUTPUTS:
        source_rows = pd.read_csv(cache_file)
        print("✓ Sentinel-2 reutilizado desde la caché tabular")
    else:
        samples = sentinel2_feature_collection(bands)
        source_rows = ee.data.computeFeatures({
            "expression": samples,
            "fileFormat": "PANDAS_DATAFRAME",
            "pageSize": 5000,
            "workloadTag": "lst-tfm-sentinel2-batch",
        })
        if not isinstance(source_rows, pd.DataFrame):
            raise RuntimeError("Earth Engine no devolvió una tabla Sentinel-2 válida.")
        temporary = cache_file.with_suffix(cache_file.suffix + ".part")
        source_rows.to_csv(temporary, index=False)
        temporary.replace(cache_file)
    if source_rows.empty:
        return pd.DataFrame(columns=["node_id", *bands, "acquisition_dt_utc", "s2_source_dt_utc"])
    source_rows["node_id"] = source_rows["node_id"].astype(str)
    source_rows["s2_source_dt_utc"] = pd.to_datetime(
        pd.to_numeric(source_rows.pop("s2_source_time_start"), errors="coerce"), unit="ms", utc=True
    )
    source_times = pd.DatetimeIndex(source_rows["s2_source_dt_utc"].dropna().unique()).sort_values()
    assigned = []
    for acquisition in acquisitions:
        if source_times.empty:
            break
        nearest = source_times[np.argmin(np.abs(source_times - acquisition))]
        if abs(nearest - acquisition) > pd.Timedelta(days=S2_MAX_DAY_DIFFERENCE):
            continue
        part = source_rows.loc[source_rows["s2_source_dt_utc"].eq(nearest)].copy()
        part["acquisition_dt_utc"] = acquisition
        assigned.append(part)
    if not assigned:
        return pd.DataFrame(columns=["node_id", *bands, "acquisition_dt_utc", "s2_source_dt_utc"])
    frame = pd.concat(assigned, ignore_index=True)
    keep = ["node_id", *[band for band in bands if band in frame], "acquisition_dt_utc", "s2_source_dt_utc"]
    return frame[keep]

node_static = static_predictors_by_node()
s2_requested_bands = s2_band_names(REQUESTED_VARIABLES)
s3_acquisition_times = sorted(pd.to_datetime(s3_pixels.acquisition_dt_utc.unique(), utc=True))
s2_nodes = download_sentinel2_batch(s3_acquisition_times, s2_requested_bands)

def aggregate_nodes_for_s3_pixel(pixel, fine):
    px, py = to_projected.transform(pixel.longitude, pixel.latitude)
    dist = np.sqrt((fine.x-px)**2 + (fine.y-py)**2)
    selected = fine[dist <= S3_NODE_AGG_RADIUS_M]
    if selected.empty:
        selected = fine.loc[[dist.idxmin()]]
    excluded = {"node_id", "x", "y", "row", "col", "acquisition_dt_utc", "s2_source_dt_utc"}
    numeric = [c for c in selected.select_dtypes(include=[np.number]).columns if c not in excluded]
    values = {f"{c}_mean_1km": selected[c].mean() for c in numeric}
    values["n_nodes_aggregated"] = len(selected)
    if "s2_source_dt_utc" in selected:
        values["s2_source_dt_utc"] = selected.s2_source_dt_utc.dropna().iloc[0] if selected.s2_source_dt_utc.notna().any() else pd.NaT
    return pd.Series(values)

aggregates = []
for acquisition, group in tqdm(s3_pixels.groupby("acquisition_dt_utc"), desc="Agregación 1 km"):
    fine = node_static.merge(s2_nodes[s2_nodes.acquisition_dt_utc == acquisition], on="node_id", how="left")
    part = group.apply(lambda pixel: aggregate_nodes_for_s3_pixel(pixel, fine), axis=1)
    part.index = group.index
    aggregates.append(part)
s3_pixels = s3_pixels.join(pd.concat(aggregates).sort_index())
display(s3_pixels.head())


## 13. ERA5-Land compartido y construcción de ambos CSV

**Propósito científico.** Este bloque aplica la siguiente etapa lógica del pipeline científico existente. Sus entradas proceden de los bloques anteriores y sus resultados alimentan la armonización o salida posterior.


In [ ]:
from tools.dataset_outputs import build_landsat_final, build_sentinel_final
from tools.sentinel3_pairing import PAIR_COLUMNS, build_landsat_sentinel_pairs

all_era5_times = pd.DatetimeIndex(pd.to_datetime(landsat_df["acquisition_dt_utc"], utc=True))
all_era5_times = all_era5_times.union(
    pd.DatetimeIndex(pd.to_datetime(s3_pixels["acquisition_dt_utc"], utc=True))
).unique().sort_values()
print(
    "Calendario meteorológico compartido:", len(all_era5_times),
    "horas objetivo para Landsat y Sentinel-3"
)
era5_hourly, era5_daily, era5_info = load_shared_era5(all_era5_times)
landsat_df = attach_era5_tables(landsat_df, "acquisition_dt_utc", era5_hourly, era5_daily)
s3_pixels = attach_era5_tables(s3_pixels, "acquisition_dt_utc", era5_hourly, era5_daily)

landsat_df = add_temporal_features(landsat_df, "acquisition_dt_utc")
landsat_df = landsat_df.sort_values(["acquisition_dt_utc", "row", "col"]).reset_index(drop=True)
landsat_df = build_landsat_final(landsat_df, REQUESTED_VARIABLES)
landsat_df.to_csv(LANDSAT_OUTPUT, index=False)

s3_pixels = add_temporal_features(s3_pixels, "acquisition_dt_utc")
s3_pixels = s3_pixels.sort_values(["acquisition_dt_utc", "s3_row", "s3_col"]).reset_index(drop=True)
s3_pixels, ndvi_filled_count = build_sentinel_final(s3_pixels, REQUESTED_VARIABLES)
s3_pixels.to_csv(SENTINEL3_OUTPUT, index=False)

paired = build_landsat_sentinel_pairs(
    landsat_df, s3_pixels, max_hour_difference=S3_PAIR_MAX_HOUR_DIFFERENCE
)
paired.to_csv(PAIRED_OUTPUT, index=False)

target_era5_hours = pd.DatetimeIndex(all_era5_times).round("h").unique()
available_era5_hours = pd.DatetimeIndex(pd.to_datetime(era5_hourly["era5_dt_utc"], utc=True)).unique()
era5_info.update({
    "target_hours": len(target_era5_hours),
    "complete_target_hours": int(target_era5_hours.isin(available_era5_hours).sum()),
    "missing_target_hours": int((~target_era5_hours.isin(available_era5_hours)).sum()),
})
source_quality = {"sentinel3": s3_quality, "era5_land": era5_info}
(OUTPUT_DIR / "calidad_fuentes.json").write_text(
    json.dumps(source_quality, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("✓ CSV Landsat:", LANDSAT_OUTPUT.resolve(), "| shape:", landsat_df.shape)
print("✓ CSV Sentinel-3:", SENTINEL3_OUTPUT.resolve(), "| shape:", s3_pixels.shape)
print("Valores NDVI completados temporalmente:", ndvi_filled_count)
print("✓ Parejas Landsat–Sentinel-3:", PAIRED_OUTPUT.resolve(), "| shape:", paired.shape)
print("Calidad Sentinel-3:", s3_quality)
print("Cobertura ERA5-Land:", era5_info)
display(landsat_df.head())
display(s3_pixels.head())


## 14. Validación final automática

**Propósito científico.** Este bloque aplica la siguiente etapa lógica del pipeline científico existente. Sus entradas proceden de los bloques anteriores y sus resultados alimentan la armonización o salida posterior.


In [ ]:
from tools.dataset_outputs import LANDSAT_PREFERRED_COLS, SENTINEL_PREFERRED_COLS
from rules.variable_catalog import output_columns_for

def validate_output(path, required, label):
    if not path.exists():
        raise AssertionError(f"No se creó {path}")
    df = pd.read_csv(path)
    if list(df.columns) != required:
        missing = [c for c in required if c not in df.columns]
        unexpected = [c for c in df.columns if c not in required]
        raise AssertionError(f"{label}: esquema incorrecto; faltan={missing}; sobran={unexpected}")
    if df.empty:
        raise AssertionError(f"{label}: el CSV está vacío")
    print(f"✓ {label}: {len(df):,} filas, {len(df.columns)} columnas")
    print("  Fechas:", df["date"].nunique(), "| rango:", df["date"].min(), "→", df["date"].max())
    print("  Nulos principales:")
    print(df[required].isna().sum().to_string())

validate_output(LANDSAT_OUTPUT, list(dict.fromkeys([*LANDSAT_PREFERRED_COLS, *output_columns_for(REQUESTED_VARIABLES, "landsat")])), "Landsat sin downscaling")
validate_output(SENTINEL3_OUTPUT, list(dict.fromkeys([*SENTINEL_PREFERRED_COLS, *output_columns_for(REQUESTED_VARIABLES, "sentinel3")])), "Sentinel-3 L2 LST a 1 km")
validate_output(PAIRED_OUTPUT, PAIR_COLUMNS, "Parejas Landsat–Sentinel-3")

print("\nProceso terminado. No se ha aplicado downscaling.")
print("Landsat:", LANDSAT_OUTPUT.resolve())
print("Sentinel-3:", SENTINEL3_OUTPUT.resolve())
print("Parejas:", PAIRED_OUTPUT.resolve())
print("Nodos:", NODES_OUTPUT.resolve())


Ejecuta de arriba abajo. Landsat, Sentinel-2, ERA5 y los lotes diarios Sentinel-3 se cachean por separado. Con `OVERWRITE_OUTPUTS=False`, una repetición reutiliza lo completado. El proceso termina con el CSV unificado, la serie Sentinel-3 y las parejas Landsat–Sentinel-3 validadas.
